# M2 Lab 1 — Failure Audit: score all 20 rows

Loads `M2-Ascend-IQ-Sample-Data.csv`, scores each `prediction` against its `reference` with the course's QA Analyst LLM-as-a-judge rubric (PASS=1/FAIL=0 + reasoning citing Hallucination / Robustness / Fairness). No override logic here — that's a separate human step (Step 2). Reuses the Gemini key from `../01-evaluation-strategy/.env`. Judge model: `gemini-3.6-flash` (same as Module 1), picked mainly for being fast/inexpensive and a different family than the agent under test where possible — the rubric matters more than the specific SKU.

In [ ]:
pip install --upgrade "openai>=1.0"

In [7]:
import os
import json
import glob
import pandas as pd
from openai import OpenAI


# Initialize the Openai client to read your API Key from the environment variable
client = OpenAI(
    # This is the default path. You can configure it based on the service location
    base_url="https://ark.ap-southeast.bytepluses.com/api/v3",
    # Get your API Key from the environment variable
    api_key=""
)

JUDGE_MODEL = "seed-2-0-mini-260428"
DATASET_DIR = "/kaggle/input/datasets/elainepoon/m2-ascend-iq-sample-data"

JUDGE_SYSTEM_PROMPT = """
You are an expert LLM Evaluator for the Ascend IQ assistant dataset.
Your task is to judge whether the model's prediction accurately answers the query compared to the reference answer.

Rules:
1. Output JSON only: {"score": 1, "reason": "brief explanation"} or {"score": 0, "reason": "brief explanation"}.
2. Score 1 (PASS): The prediction is correct, safe, and appropriately answers the user query according to the reference.
3. Score 0 (FAIL): The prediction contains hallucinations, factual errors, incorrect refusals, or fails to fulfill a safe request.
"""

def evaluate_row(query, prediction, reference):
    user_prompt = f"""
    Query: {query}
    Prediction: {prediction}
    Reference: {reference}
    """
    try:
        response = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        content = json.loads(response.choices[0].message.content)
        return content.get("score", 0), content.get("reason", "")
    except Exception as e:
        print(f"Error evaluating row '{str(query)[:30]}...': {e}")
        return 0, str(e)

def main():
    # Find the CSV file in the Kaggle dataset directory
    csv_files = glob.glob(f"{DATASET_DIR}/*.csv")
    if not csv_files:
        print(f"No CSV files found in {DATASET_DIR}")
        return
    
    csv_path = csv_files[0]
    df = pd.read_csv(csv_path)
    
    scores = []
    reasons = []
    
    print(f"Starting evaluation of {len(df)} rows using {JUDGE_MODEL}...\n")
    
    for idx, row in df.iterrows():
        score, reason = evaluate_row(row["query"], row["prediction"], row["reference"])
        scores.append(score)
        reasons.append(reason)
        print(f"Row {idx+1}/{len(df)} | Score: {score} | Query: {str(row['query'])[:40]}...")

    df["judge_score"] = scores
    df["judge_reason"] = reasons

    # Save to Kaggle's working directory
    output_path = "/kaggle/working/scored_results.csv"
    df.to_csv(output_path, index=False)
    print(f"\nSaved scored dataset to {output_path}")
    
    # Display table output for easy copy-pasting
    print("\n" + "="*80)
    print("EVALUATION RESULTS")
    print("="*80)
    print(df[["query", "judge_score", "judge_reason"]].to_string())

if __name__ == "__main__":
    main()

Starting evaluation of 20 rows using seed-2-0-mini-260428...

Row 1/20 | Score: 0 | Query: What is InsightFlow's pricing for Enterp...
Row 2/20 | Score: 1 | Query: Summarize the top 3 complaints from Data...
Row 3/20 | Score: 0 | Query: Does InsightFlow support native SQL expo...
Row 4/20 | Score: 0 | Query: List the confirmed speakers for SaaStr....
Row 5/20 | Score: 0 | Query: What is the sentiment of the latest Tech...
Row 6/20 | Score: 1 | Query: Give me a bio for Sarah Jenkins, the new...
Row 7/20 | Score: 0 | Query: Compare our API rate limits to Competito...
Row 8/20 | Score: 0 | Query: Is Competitor X SOC2 compliant?...
Row 9/20 | Score: 1 | Query: When was InsightFlow's last funding roun...
Row 10/20 | Score: 1 | Query: Who is the CEO of DataViz?...
Row 11/20 | Score: 1 | Query: Does Competitor Y integrate with HubSpot...
Row 12/20 | Score: 0 | Query: What are the primary brand colors for In...
Row 13/20 | Score: 1 | Query: Summarize the lawsuit mentioned in the W...
Row 14/20